In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle

# silence warnings
import warnings
warnings.filterwarnings('ignore')

# binning
try:
    from optbinning import OptimalBinning
except:
    ! pip install optbinning
    from optbinning import OptimalBinning

(CVXPY) Dec 17 05:59:25 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Dec 17 05:59:25 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')


#### Functions

In [2]:
def bin_features(list_cols, X, y, dict_monotone_constraints, int_n_bins):
    dict_bins = {}
    list_cols_scorecard = []
    for col in tqdm(list_cols):
        # copy the original
        int_n_bins_tmp = int_n_bins
        
        # get the monotonicity
        int_monotonicity = dict_monotone_constraints[col]
        # logic
        if int_monotonicity == -1:
            str_monotonic_trend = 'descending'
        else:
            str_monotonic_trend = 'ascending'
        
        # make sure maximum number of bins are showing
        while True:
            # init
            cls_binning = OptimalBinning(
                name=col,
                dtype='numerical',
                solver='cp',
                monotonic_trend=str_monotonic_trend,
                min_n_bins=int_n_bins_tmp,
                max_n_bins=int_n_bins_tmp,
            )
            # fit
            cls_binning.fit(
                X[col],
                y,
            )
            # assign
            dict_bins[col] = cls_binning

            # transform
            str_col = f'{col}_binned'
            X[str_col] = cls_binning.transform(
                X[col],
                metric='woe',
            )

            # get count unique
            int_n_unique = X[str_col].nunique()
            
            # logic
            if int_n_unique == int_n_bins_tmp:
                break
            else:
                # subtract
                int_n_bins_tmp -= 1

        # append
        list_cols_scorecard.append(str_col)
    # return
    return dict_bins, list_cols_scorecard, X

#### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_subtask = os.getcwd().split('/')[6]
print(f'Subtask: {str_subtask}')

str_target = 'Early_Pay_Delinquency_15_60_Flag'
print(f'Target: {str_target}')

str_dirname_output = './output'

str_task_tmp = '09_15_in_60'

Project: 20241112-simple-model-test
Task: create_grid
Subtask: 01_15_in_60
Target: Early_Pay_Delinquency_15_60_Flag


#### Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [5]:
str_filename = 'df_holdout.gzip'
str_uri = f's3://{str_project}/{str_task_tmp}/01_data_split/{str_filename}'
df = pd.read_parquet(
    str_uri,
)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,ENG-payment_to_income,ENG-loan_to_value,ENG-bk,request_month,train,valid,test,inform,holdout,data_set
87591,7996114,2024-07-27 00:59:37+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-07-26.gzip,1,1,Nevada,Franchise,Nevada,False,...,NaN,1.021137,0,2024-07-01,0,0,0,0,1,holdout
86722,7983349,2024-07-15 21:05:20+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-07-15.gzip,1,1,Maryland,Independent,Maryland,False,...,NaN,1.437162,0,2024-07-01,0,0,0,0,1,holdout
86667,7983975,2024-07-12 23:35:06+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-07-12.gzip,1,1,Oregon,Franchise,Oregon,True,...,NaN,1.192722,0,2024-07-01,0,0,0,0,1,holdout
86958,7996439,2024-07-18 01:36:07+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-07-17.gzip,1,1,Indiana,Independent,Indiana,True,...,0.04635,1.351321,0,2024-07-01,0,0,0,0,1,holdout
86787,7996446,2024-07-16 03:43:17+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-07-15.gzip,1,1,Ohio,Franchise,Ohio,True,...,NaN,1.283317,1,2024-07-01,0,0,0,0,1,holdout
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89597,8173711,2024-09-04 03:28:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-09-03.gzip,1,1,Maryland,Independent,Maryland,False,...,NaN,1.229124,0,2024-09-01,0,0,0,0,1,holdout
90088,8171521,2024-09-12 03:38:11+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-09-11.gzip,0,0,California,Franchise,California,True,...,NaN,1.082279,0,2024-09-01,0,0,0,0,1,holdout
89563,8173663,2024-09-03 22:20:34+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-09-03.gzip,0,0,Idaho,Franchise,Idaho,True,...,NaN,1.036325,1,2024-09-01,0,0,0,0,1,holdout
89749,8174035,2024-09-06 04:10:05+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-09-05.gzip,0,0,Arizona,Franchise,Arizona,False,...,NaN,1.402660,1,2024-09-01,0,0,0,0,1,holdout


#### List of columns

In [6]:
list_cols = [
    'fltgrossmonthly__income_sum',
    'ENG-loan_to_value',
    'ENG-payment_to_income',
    'dti__app',
    'flt_avg_open__tu_pmthx',
    'flt_avg_closed__tu_pmthx',
    'flt_wtd_avg_open__tu_pmthx',
    'flt_wtd_avg_closed__tu_pmthx',
]

#### Impute

In [7]:
str_filename = 'dict_impute.pkl'
str_local_path = f'../../{str_task_tmp}/02_model/output/{str_filename}'
dict_impute = pickle.load(open(str_local_path, 'rb'))

# impute
for key, val in tqdm(dict_impute.items()):
    df[key] = df[key].fillna(val)

100%|██████████| 50/50 [00:00<00:00, 2955.28it/s]


#### Monotonic constraints

In [8]:
# get monotonic constraints
dict_monotone_constraints = {
    'fltgrossmonthly__income_sum': -1,
    'ENG-loan_to_value': 1,
    'ENG-payment_to_income': 1,
    'dti__app': 1,
    'flt_avg_open__tu_pmthx': -1,
    'flt_avg_closed__tu_pmthx': -1,
    'flt_wtd_avg_open__tu_pmthx': -1,
    'flt_wtd_avg_closed__tu_pmthx': -1,
}
dict_bins, list_cols_scorecard, X = bin_features(
    list_cols=list_cols,
    X=df[list_cols].copy(),
    y=df[str_target],
    dict_monotone_constraints=dict_monotone_constraints,
    int_n_bins=5,
)

100%|██████████| 8/8 [00:01<00:00,  6.42it/s]


In [9]:
# show bins
df_tmp = X.copy()
df_tmp['target'] = df[str_target]
list_df = []
for col in tqdm(list_cols_scorecard):
    # get original column
    str_col_original = col.split('_binned')[0]
    # get min max and mean
    df_grouped = df_tmp.groupby(by=col, as_index=False).agg({
        str_col_original: ['min', 'max', 'count'],
        'target': 'mean',
    })
    # sort
    df_grouped.sort_values(by=(str_col_original, 'min'), ascending=True, inplace=True)
    df_grouped['feature'] = col
    # rename
    list_cols = ['bin','min','max','count','target','feature']
    df_grouped.columns = list_cols
    df_grouped['prop'] = df_grouped['count'] / df_grouped['count'].sum()
    # bin label
    df_grouped['bin_label'] = range(1, df_grouped.shape[0]+1)
    # reorder
    list_cols = ['feature','bin','min','max','count','target','bin_label']
    df_grouped = df_grouped[list_cols].copy()
    # append
    list_df.append(df_grouped)

# make df
df_bins = pd.concat(list_df)

# save
str_filename = 'df_bins.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_bins.to_csv(str_local_path, index=False)

# show
df_bins

100%|██████████| 8/8 [00:00<00:00, 121.29it/s]


,feature,bin,min,max,count,target,bin_label
0,fltgrossmonthly__income_sum_binned,-0.086400,0.000000,4679.000000,2189,0.129283,1
1,fltgrossmonthly__income_sum_binned,-0.065574,4680.000000,5199.000000,575,0.126957,2
2,fltgrossmonthly__income_sum_binned,0.057223,5200.000000,6593.000000,1132,0.113958,3
3,fltgrossmonthly__income_sum_binned,0.162448,6594.000000,8286.100000,1118,0.103757,4
1,ENG-loan_to_value_binned,0.228589,0.324375,1.044339,358,0.097765,1
0,ENG-loan_to_value_binned,-0.016009,1.045152,2.190800,4656,0.121564,2
3,ENG-payment_to_income_binned,0.369495,0.015980,0.064442,279,0.086022,1
2,ENG-payment_to_income_binned,0.161417,0.064472,0.106123,597,0.103853,2
1,ENG-payment_to_income_binned,0.051041,0.106279,0.177983,480,0.114583,3
0,ENG-payment_to_income_binned,-0.054661,0.178297,402.000000,3658,0.125752,4
